# 03 · Fitting

How well the body and head models fit the videos, and the avatar mesh joining them. Run `avatar-splats fit` first; this notebook only reads its results.

The overlays compare the meshes' own outlines with the masks: white where the mask and the fit agree, red where only the mask covers the pixel, cyan where only the fit does.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import plotly.graph_objects as go
import torch
from PIL import Image

from avatar_splats.body_models import hybrid
from avatar_splats.body_models.anny import AnnyBody
from avatar_splats.body_models.ict import SKIN, IctFaceKit
from avatar_splats.config import load_config
from avatar_splats.fitting import HEAD_VIDEOS, BodyFit, HeadFit, fitted_path, head_vertices
from avatar_splats.fitting.body import fitted_body
from avatar_splats.fitting.silhouette import overlay
from avatar_splats.preprocess import Preprocessed, preprocessed_path
from avatar_splats.rendering import rasterize

config = load_config("../configs/default.yaml")
device = config.fitting.device
body_fit = BodyFit.load(fitted_path(config, "body"))
head_fits = {video: HeadFit.load(fitted_path(config, video)) for video in HEAD_VIDEOS}
avatar = hybrid.Avatar.load(fitted_path(config, "avatar"))

## Per-frame fit

The consistency diagnostic: frames where the silhouette or keypoints fit worse than their neighbours are where the video disagrees with a single consistent body or head.

In [ ]:
fig, (iou_ax, error_ax) = plt.subplots(2, 1, figsize=(10, 6), sharex=True)
iou_ax.plot(body_fit.frame_indices, body_fit.silhouette_iou, label="body")
error_ax.plot(body_fit.frame_indices, body_fit.keypoint_error, label="body keypoints")
for video, fit in head_fits.items():
    iou_ax.plot(fit.frame_indices, fit.silhouette_iou, label=video)
    error_ax.plot(fit.frame_indices, fit.landmark_error, label=f"{video} landmarks")
iou_ax.set_ylabel("silhouette IoU")
error_ax.set_ylabel("error (pixels)")
error_ax.set_xlabel("frame")
iou_ax.legend()
error_ax.legend()
fig.tight_layout()

## Body

In [ ]:
body = AnnyBody().to(device)
frames = list(range(0, len(body_fit.frame_indices), 10))
camera = body_fit.camera().scaled(0.4).to(device)
with torch.no_grad():
    _, vertices = fitted_body(body_fit, body, frames)
    rendered = torch.stack([
        rasterize(camera.screen(v), body.faces, camera.width, camera.height).mask().float()
        for v in vertices
    ])  # fmt: skip
masks = Preprocessed.load(preprocessed_path(config, "body")).masks
tiles = [
    overlay(np.array(Image.fromarray(masks[f]).resize((camera.width, camera.height))), r)
    for f, r in zip(frames, rendered.cpu().numpy(), strict=True)
]
plt.figure(figsize=(18, 5))
plt.imshow(np.concatenate(tiles, axis=1))
plt.axis("off");

In [ ]:
print(
    {
        label: round(float(value), 2)
        for label, value in zip(body.phenotype_labels, body_fit.phenotype, strict=True)
    }
)
# The local changes the fit used most.
changes = sorted(
    zip(body.local_change_labels, body_fit.local_changes, strict=True), key=lambda c: -abs(c[1])
)
print({label: round(float(value), 2) for label, value in changes[:15]})

## Heads

In [ ]:
head = IctFaceKit(config.body_models.ict_facekit).to(device)
skin = head.faces[head.faces.max(dim=1).values < SKIN.stop]
fig, axes = plt.subplots(len(head_fits), 1, figsize=(18, 4 * len(head_fits)))
for ax, (video, fit) in zip(axes, head_fits.items(), strict=True):
    frames = list(range(0, len(fit.frame_indices), 10))
    camera = fit.camera().scaled(0.25).to(device)

    def tensor(array):
        return torch.from_numpy(array).float().to(device)

    with torch.no_grad():
        vertices = head_vertices(
            head,
            tensor(fit.identity)[None],
            tensor(fit.expressions[frames]),
            tensor(fit.rotations[frames]),
            tensor(fit.translations[frames]),
            tensor(fit.centre),
        )
        rendered = torch.stack([
            rasterize(camera.screen(v), skin, camera.width, camera.height).mask().float()
            for v in vertices
        ])  # fmt: skip
    masks = Preprocessed.load(preprocessed_path(config, video)).masks
    tiles = [
        overlay(np.array(Image.fromarray(masks[f]).resize((camera.width, camera.height))), r)
        for f, r in zip(frames, rendered.cpu().numpy(), strict=True)
    ]
    ax.imshow(np.concatenate(tiles, axis=1))
    ax.set_title(video, loc="left")
    ax.axis("off")

The strongest expressions found in the expressions video:

In [ ]:
fit = head_fits["head_expressions"]
strongest = np.argsort(-fit.expressions.max(axis=0))[:8]
plt.figure(figsize=(10, 4))
for i in strongest:
    plt.plot(fit.frame_indices, fit.expressions[:, i], label=head.expression_names[i])
plt.xlabel("frame")
plt.ylabel("weight")
plt.legend(ncol=2);

## The avatar mesh

Anny's body (grey) with the ICT-FaceKit head: skin (orange), mouth interior (red) and eyeballs (blue). Drag to turn it.

In [ ]:
colours = np.array(["#9a9a9a", "#e07b39", "#c0392b", "#2e86de"])[avatar.face_parts]
x, y, z = avatar.rest_vertices.T
i, j, k = avatar.faces.T
figure = go.Figure(go.Mesh3d(x=x, y=y, z=z, i=i, j=j, k=k, facecolor=colours, flatshading=True))
figure.update_layout(scene_aspectmode="data", height=700, margin=dict(l=0, r=0, t=0, b=0))
figure.show()